# 05 - ChemBERTa (Pretrained Transformer)
## Predicting Human Intestinal Absorption (HIA)
**Student:** Mohammad Karamath Fardeen | ID: 25251265  
**Supervisor:** Kolawole Adebayo | Maynooth University | 2025-2026

Model: ChemBERTa (Chithrananda et al., 2020)  
RoBERTa-based Transformer pretrained on 77 million SMILES strings from PubChem/ZINC

In [ ]:
# !pip install transformers
import os, numpy as np, pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import get_linear_schedule_with_warmup
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, matthews_corrcoef
from tqdm import tqdm

SEED = 42
torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MODEL_NAME = 'seyonec/ChemBERTa-zinc-base-v1'
print(f'Device: {DEVICE}')

## 1. Dataset and Tokenizer

In [ ]:
class HIASmilesDataset(Dataset):
    def __init__(self, csv_path, tokenizer, max_length=128):
        df = pd.read_csv(csv_path)
        self.smiles = df['Drug'].tolist()
        self.labels = df['Y'].astype(int).tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self): return len(self.smiles)

    def __getitem__(self, idx):
        enc = self.tokenizer(self.smiles[idx], max_length=self.max_length,
                             padding='max_length', truncation=True, return_tensors='pt')
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'label':          torch.tensor(self.labels[idx], dtype=torch.long)
        }

print(f'Loading ChemBERTa tokenizer from: {MODEL_NAME}')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_ds = HIASmilesDataset('data/raw/hia_train.csv', tokenizer)
valid_ds = HIASmilesDataset('data/raw/hia_valid.csv', tokenizer)
test_ds  = HIASmilesDataset('data/raw/hia_test.csv',  tokenizer)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
valid_loader = DataLoader(valid_ds, batch_size=16, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=16, shuffle=False)
print(f'Train: {len(train_ds)} | Valid: {len(valid_ds)} | Test: {len(test_ds)}')

## 2. Fine-tune ChemBERTa

In [ ]:
print('Loading ChemBERTa model...')
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2, ignore_mismatched_sizes=True).to(DEVICE)

# Class weights for imbalance
labels = [d['label'].item() for d in train_ds]
n_pos, n_neg = sum(labels), len(labels) - sum(labels)
class_weights = torch.tensor([1.0, n_neg/max(n_pos,1)], dtype=torch.float).to(DEVICE)
criterion = torch.nn.CrossEntropyLoss(weight=class_weights)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
total_steps = len(train_loader) * 20
scheduler = get_linear_schedule_with_warmup(optimizer,
                num_warmup_steps=int(0.1*total_steps),
                num_training_steps=total_steps)

def evaluate(model, loader):
    model.eval()
    all_preds, all_probs, all_labels = [], [], []
    with torch.no_grad():
        for batch in loader:
            ids   = batch['input_ids'].to(DEVICE)
            mask  = batch['attention_mask'].to(DEVICE)
            logits = model(input_ids=ids, attention_mask=mask).logits
            probs  = torch.softmax(logits, dim=-1)[:,1].cpu().numpy()
            preds  = (probs > 0.5).astype(int)
            all_probs.extend(probs)
            all_preds.extend(preds)
            all_labels.extend(batch['label'].numpy())
    return {
        'roc_auc':  round(roc_auc_score(all_labels, all_probs), 4),
        'f1':       round(f1_score(all_labels, all_preds), 4),
        'accuracy': round(accuracy_score(all_labels, all_preds), 4),
        'mcc':      round(matthews_corrcoef(all_labels, all_preds), 4),
    }

best_val_auc, best_state, patience_counter = 0.0, None, 0

print('Fine-tuning ChemBERTa...')
for epoch in range(1, 21):
    model.train()
    total_loss = 0
    for batch in tqdm(train_loader, desc=f'Epoch {epoch}/20', leave=False):
        ids    = batch['input_ids'].to(DEVICE)
        mask   = batch['attention_mask'].to(DEVICE)
        labels = batch['label'].to(DEVICE)
        optimizer.zero_grad()
        logits = model(input_ids=ids, attention_mask=mask).logits
        loss   = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    val_metrics = evaluate(model, valid_loader)
    print(f'Epoch {epoch:2d} | Loss: {total_loss/len(train_loader):.4f} | Val AUC: {val_metrics["roc_auc"]:.4f}')

    if val_metrics['roc_auc'] > best_val_auc:
        best_val_auc = val_metrics['roc_auc']
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= 5:
            print(f'Early stopping at epoch {epoch}')
            break

model.load_state_dict(best_state)
test_metrics = evaluate(model, test_loader)
print(f'\nChemBERTa Test Results:')
for k, v in test_metrics.items():
    print(f'  {k}: {v}')